In [ ]:
import argparse
import gc
from typing import List, Union
import os
import pandas as pd
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
import torch

In [ ]:
!nvidia-smi

import torch
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda if torch.cuda.is_available() else 'Not available'}")

gc.collect()
torch.cuda.empty_cache()

In [ ]:
model_name = "Qwen/Qwen2.5-7B-Instruct"
input_csv = "train_harmful_prompts.csv"
output_repetitions = 1
max_new_tokens = 2048
temperature = 0.6
batch_size = 1
tensor_parallel_size = 1
gpu_memory_utilization = 0.9

In [ ]:
def read_csv(input_csv: str) -> Union[List[str], None]:
    """Read prompts from CSV file."""
    print(f"Reading prompts from {input_csv}...")
    try:
        df = pd.read_csv(input_csv)
        df = df.head(3)
        prompts = df['prompt'].tolist()
        print(f"Loaded {len(prompts)} prompts")
        return prompts
    except Exception as e:
        print(f"Error reading CSV: {e}")
        return None


def save_csv(results: List[dict], output_csv: str):
    """Save results to CSV."""
    print(f"\nSaving {len(results)} results to {output_csv}...")
    try:
        output_df = pd.DataFrame(results)
        print(output_df)
        # output_df.to_csv(output_csv, index=False)
        print(f"Results saved successfully to {output_csv}")
    except Exception as e:
        print(f"Error saving CSV: {e}")


def apply_chat_template_batch(prompts: List[str], tokenizer) -> List[str]:
    """Apply chat template to batch of prompts."""
    formatted_prompts = []
    for prompt in prompts:
        chat = [{"role": "user", "content": prompt}]
        formatted_prompts.append(tokenizer.apply_chat_template(
            chat, add_generation_prompt=True, tokenize=False
        ))
    return formatted_prompts


def generate_outputs(
    llm: LLM,
    tokenizer,
    prompts: List[str],
    sampling_params: SamplingParams,
    output_repetitions: int,
    batch_size: int
) -> List[dict]:
    """Generate multiple output variations for each prompt."""
    print(f"\nGenerating {output_repetitions} outputs for {len(prompts)} prompts")
    print(f"Total generations: {len(prompts) * output_repetitions}")

    all_results = []

    for batch_start in range(0, len(prompts), batch_size):
        batch_end = min(batch_start + batch_size, len(prompts))
        batch_prompts = prompts[batch_start:batch_end]

        print(f"\nProcessing batch: prompts {batch_start+1}-{batch_end}")

        # Create repeated prompts for output repetitions
        repeated_prompts = []
        prompt_indices = []

        for i, prompt in enumerate(batch_prompts):
            for rep in range(output_repetitions):
                repeated_prompts.append(prompt)
                prompt_indices.append(i)

        # Apply chat template
        formatted_prompts = apply_chat_template_batch(repeated_prompts, tokenizer)

        print("formatted_prompts: ", formatted_prompts)

        # Generate
        outputs = llm.generate(formatted_prompts, sampling_params)

        # Process outputs
        for i, output in enumerate(outputs):
            original_idx = prompt_indices[i]
            original_prompt = batch_prompts[original_idx]
            output_rep = (i % output_repetitions) + 1

            generated_text = tokenizer.decode(output.outputs[0].token_ids, skip_special_tokens=True)

            all_results.append({
                "prompt": original_prompt,
                "output": generated_text,
                "output_rep_n": output_rep
            })

        gc.collect()

    print(f"\nGeneration complete: {len(all_results)} total outputs")
    return all_results

In [ ]:
print(f"CUDA available: {torch.cuda.is_available()}")
gc.collect()
torch.cuda.empty_cache()

# Handle CSV file selection
input_csv = os.path.join('../dataset', input_csv)

# Read prompts
prompts = read_csv(input_csv)

# Set up sampling parameters
sampling_params = SamplingParams(
    max_tokens=max_new_tokens,
    temperature=temperature,
)

# Initialize tokenizer
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

# Initialize vLLM
print("Initializing vLLM...")
llm = LLM(
    model=model_name,
    tensor_parallel_size=tensor_parallel_size,
    gpu_memory_utilization=gpu_memory_utilization,
    trust_remote_code=True,
)
print("Model loaded successfully!")

# Generate outputs
results = generate_outputs(llm, tokenizer, prompts, sampling_params, output_repetitions, batch_size)

# Construct output CSV path
input_csv_name = os.path.splitext(input_csv)[0]
output_dir = os.path.join('results', model_name, 'dataset')
output_csv = os.path.join(output_dir, f'{input_csv_name}_out{output_repetitions}.csv')

# Save results
save_csv(results, output_csv)

# Cleanup
del llm, tokenizer
gc.collect()
torch.cuda.empty_cache()

print(f"\n{'='*60}")
print(f"✅ Processing complete")
print(f"   Prompts: {len(prompts)}")
print(f"   Outputs: {len(results)}")
print(f"   Saved to: {output_csv}")
print(f"{'='*60}")